In [0]:
from pyspark.sql import functions as F

def json_to_df(s):
    """Turn one JSON string into a DataFrame, inferring the schema."""
    return (spark.createDataFrame([(s,)], ["v"])
            .select(F.from_json("v", F.schema_of_json(F.lit(s))).alias("j"))
            .select("j.*"))

####Exercise 1A: Flat JSON

In [0]:
s1 = '''{"order_id": 1001, "customer_name": "Anna Schmidt", "amount": 249.99, "is_paid": true, "order_date": "2026-03-14"}'''

In [0]:
df = json_to_df(s1)
# display(df)

# df.printSchema()

df = (df
      .withColumn("order_date", F.to_date("order_date"))
      .withColumn("amount", F.col("amount").cast("decimal(10,2)")
      )
    )
display(df)
# df.printSchema()

#### Exercise 1B: Nested object (struct)

In [0]:
s2 = '''{
  "order_id": 1002,
  "customer": {"id": "C17", "name": "Lukas Meyer", "city": "Berlin"},
  "shipping": {"method": "DHL", "address": {"street": "Hauptstr. 5", "zip": "10115"}}
}'''

df = json_to_df(s2)
# df.printSchema()

df_flat = df.select(
    "order_id",
    F.col("customer.id").alias("customer_id"),
    F.col("customer.name").alias("customer_name"),
    F.col("customer.city").alias("city"),
    F.col("shipping.method").alias("shipping_method"),
    F.col("shipping.address.street").alias("street"),
    F.col("shipping.address.zip").alias("zip")
    )
df_flat.show()

####Exercise 1C: Array of simple values

In [0]:
s3 = '''{"order_id": 1003, "customer": "Mia Fischer", "tags": ["new", "vip", "newsletter"], "scores": [4, 5, 3]}'''

In [0]:
df = json_to_df(s3)
# df.printSchema()

df.select("order_id",
          F.size("tags").alias("tag_count"),
          F.col("tags")[0].alias("first_tag"),
          F.array_contains("tags","vip").alias("is_vip"),
          F.aggregate("scores",F.lit(0).cast("bigint"), lambda acc, x : acc + x).alias("score_sum")
          ).show()
          

In [0]:
# average: sum / size

# df.select(F.aggregate(("scores"),F.lit(0.0), lambda a,x : a + x)/F.size("scores")).alias("average").show()

# one row per tag
df.select("order_id",F.explode("tags").alias("tags")).show()

####Exercise 1D: Multiple records in a file (a very common real-world gotcha)

In [0]:
jsonl = '''{"order_id": 1, "amount": 10.5}
{"order_id": 2, "amount": 20.0}
{"order_id": 3, "amount": 5.25}'''

json_array = '''[
  {"order_id": 1, "amount": 10.5},
  {"order_id": 2, "amount": 20.0},
  {"order_id": 3, "amount": 5.25}
]'''

# dbutils.fs.put("/tmp/jsonl_demo/orders.jsonl", jsonl, True)
# dbutils.fs.put("/tmp/jsonl_demo/orders_array.json", json_array, True)

In [0]:
# spark.read.json("/tmp/jsonl_demo/orders.jsonl").show()       # works: one record per line

spark.read.json("/tmp/jsonl_demo/orders_array.json").show()  # shows _corrupt_record, not your data

# spark.read.option("multiLine", True).json("/tmp/jsonl_demo/orders_array.json").show()   # fixed